In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import densenet121
from torchvision import transforms
from PIL import Image

def predict_knee_xray(image_path, model_path="Models/best_densenet121_binary.pth", threshold=0.50):
    """
    Loads the trained DenseNet121 model and predicts the KL grade binary group.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = densenet121()
    num_ftrs = model.classifier.in_features 
    
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5, inplace=False), # inplace=False is safer for inference
        nn.Linear(in_features=num_ftrs, out_features=2, bias=True) 
    )
    
    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print("[*] Model weights loaded successfully.")
    except FileNotFoundError:
        print(f"[!] Error: Could not find '{model_path}'. Ensure the path is correct.")
        return
        
    model = model.to(device)
    model.eval() 

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3), 
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    try:
        image = Image.open(image_path).convert("RGB") 
    except Exception as e:
        print(f"[!] Error loading image: {e}")
        return
        
    input_tensor = val_transforms(image).unsqueeze(0).to(device) 
    

    with torch.no_grad():
        outputs = model(input_tensor)
        
        probabilities = F.softmax(outputs, dim=1)
        prob_positive = probabilities[0, 1].item()

    predicted_class = 1 if prob_positive >= threshold else 0
    
    if predicted_class == 1:
        diagnosis = "Positive/Definite (Grade 2, 3, or 4)"
    else:
        diagnosis = "Negative/Doubtful (Grade 0 or 1)"

        
    print("\n" + "=" * 40)
    print(f" INFERENCE RESULTS")
    print("=" * 40)
    print(f"File: {image_path}")
    print(f"Osteoarthritis Probability: {prob_positive * 100:.2f}%")
    print(f"Diagnosis (Threshold={threshold}): {diagnosis}")
    print("=" * 40 + "\n")
    
    return predicted_class, prob_positive

    
if __name__ == "__main__":

    test_image = "Data/test/4/9504935L.png"
    
    # You can tweak the threshold here without retraining, just like in the testing loop!
    
    predict_knee_xray(test_image, model_path="Models/best_densenet121_binary.pth", threshold=0.59)

Using device: cpu
[*] Model weights loaded successfully.

 INFERENCE RESULTS
File: Data/test/4/9504935L.png
Osteoarthritis Probability: 92.59%
Diagnosis (Threshold=0.59): Positive/Definite (Grade 2, 3, or 4)



In [28]:
!pip install streamlit

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl.metadata (14 kB)
  Using cached pydeck-0.9.2-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.2 MB 3.4 MB/s eta 0:00:03
   --------- ------------------------------ 2.1/9.2 MB 5.3 MB/s eta 0:00:02
   ------------------- -------------------- 4.5/9